# MFP IFPA WPPR Projections
## PER https://www.ifpapinball.com/menu/ranking-info/
* The value of a tournament is calculated from determining a Base Value, a Tournament Value Adjustment (TVA) based on the quality of the players participating, and a Tournament Grading Percentage (TGP) based on the quality of the format of the tournament. Full details on each of those metrics are available below.

In [1]:
import pandas as pd
import numpy as np
import requests

In [2]:
api_key = '90c34496d6eb32438ebe19c9a993327d'

In [3]:
player_search_url = 'https://api.ifpapinball.com/v1/player/search?api_key='+ api_key +'&q='

In [4]:
# search for each player
def ifpa_player_search(row):
    player_name = row['name']
    r = requests.get(player_search_url+player_name)
    try: 
        data = r.json()
    
        # check if there are any resuls
        if data['search'] == 'No players found':
            return row
        # if so, assume the player is the first listed
        else:
            for item in data['search'][0].items():
                row[item[0]] = item[1]
        #change rank to int
        row['wppr_rank'] = int(row['wppr_rank'])
        return row
    except:
        return row

In [5]:
series_id = '2456'

In [6]:
mfp_df = pd.read_csv('mfp_league_'+series_id+'_standings.csv')

In [7]:
num_weeks = mfp_df.weeks_played.max()
weeks_left = 10 - num_weeks

In [8]:
mfp_df = mfp_df[mfp_df.weeks_played + weeks_left > 2]

In [9]:
mfp_df.rename(index = str, columns = {'player_id' : 'name'}, inplace = True)

In [10]:
mfp_df.name = mfp_df.name.str.translate("“”")

In [11]:
mfp_df.name = mfp_df.name.str.replace("BeeOtch","Bee")

In [12]:
mfp_df = mfp_df.apply(ifpa_player_search, axis=1)

In [13]:
mfp_df.loc[mfp_df.name == 'RC', 'player_id'] = '60584'
mfp_df.loc[mfp_df.name == 'Iris', 'player_id'] = '63908'


mfp_df.loc[mfp_df.name == 'RC', 'wppr_rank'] = 6283
mfp_df.loc[mfp_df.name == 'Iris', 'wppr_rank'] = 9847

# Nikki changed her name week 10 winter 19
mfp_df.loc[mfp_df.name == 'Nikki Carmichael', 'player_id'] = '36495'
mfp_df.loc[mfp_df.name == 'Nikki Carmichael', 'wppr_rank'] = 3497

In [14]:
mfp_df.loc[mfp_df.name == 'RC', 'player_id']

Series([], Name: player_id, dtype: object)

In [15]:
player_id_url = 'https://api.ifpapinball.com/v1/player/'
api_url = '?api_key=' + api_key

In [16]:
def ifpa_player_id(row):
    if pd.isna(row['player_id']):
        return row
    player_id = row['player_id']
    
    
    
    r = requests.get(player_id_url+str(player_id)+api_url)
    data = r.json()
    
    try:
        for item in data['player_stats'].items():
            row[item[0]] = item[1]
    except:
        return row
    #change rank to int
    #row['wppr_rank'] = int(row['wppr_rank'])
    return row

In [17]:
mfp_df = mfp_df.apply(ifpa_player_id, axis=1)

In [18]:
mfp_df.columns

Index(['91361', '91363', '91364', '91365', '91366', '91367', '91368', '91370',
       '91371', '91372', 'average_finish', 'average_finish_last_year',
       'avg_att', 'avg_score', 'best_finish', 'best_finish_count', 'city',
       'country_code', 'country_name', 'cur_result', 'cur_total',
       'current_score', 'current_wppr_rank', 'current_wppr_value',
       'efficiency_rank', 'efficiency_value', 'first_name', 'highest_rank',
       'highest_rank_date', 'kept_score', 'kept_score_att', 'last_month_rank',
       'last_name', 'last_year_rank', 'league_rank', 'max_score_possible',
       'mean_adj_score', 'mean_adj_score_att', 'name', 'player_id', 'proj_att',
       'proj_score', 'proj_score_att', 'proj_type', 'ratings_rank',
       'ratings_value', 'remaining_weeks_above_avg', 'required_avg_trophy',
       'required_avg_win', 'score_rank', 'score_to_beat', 'state',
       'top_score_1', 'top_score_2', 'top_score_3', 'top_score_4',
       'top_score_5', 'top_score_6', 'total_active_eve

In [19]:
mfp_df.total_events_all_time = pd.to_numeric(mfp_df.total_events_all_time)
mfp_df.ratings_value = pd.to_numeric(mfp_df.ratings_value)
mfp_df.current_wppr_value = pd.to_numeric(mfp_df.current_wppr_value)

In [20]:
if max(mfp_df.weeks_played == 10):
    mfp_df.proj_score = mfp_df.current_score

In [21]:
mfp_df.head()

,91361,91363,91364,91365,91366,91367,91368,91370,91371,91372,...,top_score_5,top_score_6,total_active_events,total_events_all_time,total_events_away,weeks_left_att,weeks_played,weeks_won,wppr_points_all_time,wppr_rank
0,29.0,27.0,25.0,33.0,23.0,27.0,25.0,19.0,33.0,21.0,...,27.0,25.0,26,84.0,0,0,10,2,236.19,1006.0
1,NaN,23.0,35.0,25.0,16.0,27.0,29.0,NaN,24.0,29.0,...,25.0,24.0,47,136.0,0,0,8,2,753.21,355.0
2,24.0,27.0,29.0,29.0,21.0,21.0,30.0,NaN,27.0,25.0,...,27.0,25.0,57,200.0,127,0,9,0,916.92,244.0
3,25.0,27.0,27.0,21.0,25.0,29.0,21.0,31.0,25.0,19.0,...,25.0,25.0,43,88.0,0,0,10,2,456.25,311.0
4,31.0,23.0,27.0,17.0,25.0,27.0,23.0,24.0,21.0,27.0,...,25.0,24.0,25,48.0,48,0,10,1,157.23,1384.0


In [22]:
#### mfp_df.to_csv('mfp_season_df_ifpa.csv')

In [23]:
# Base Points
base_points = mfp_df.loc[mfp_df.total_events_all_time >= 5].shape[0] * 0.5
if base_points > 32:
    base_points = 32
    
base_points

23.0

In [24]:
# TVA
# TVA Rating
mfp_df['tva_rating'] = mfp_df.ratings_value[mfp_df.ratings_value > 1285.71]*0.000546875 - 0.703125
tva_rating = sum(mfp_df.ratings_value[mfp_df.ratings_value > 1285.71]*0.000546875 - 0.703125)
if tva_rating > 25:
    tva_rating = 25
    
tva_rating

2.135825000000002

In [25]:
# TVA
# TVA Ranking
mfp_df['tva_points'] = np.log(mfp_df.wppr_rank)*(-0.211675054) + 1.459827968
tva_points = sum(mfp_df.tva_points[mfp_df.tva_points > 0])
tva_points

0.9426987488672229

In [26]:
mfp_df.loc[mfp_df.tva_points < 0, 'tva_points'] = 0

In [27]:
mfp_df['tva_total'] = mfp_df.tva_rating + mfp_df.tva_points

In [28]:
total_before_tgp = base_points + tva_rating + tva_points

In [29]:
tgp = 100
first_place_value = (tgp/100)* total_before_tgp
first_place_value

26.078523748867223

In [30]:
# Qualifying Players
#mfp_df.loc[(mfp_df.total_events_all_time >= 5) & (mfp_df.num_games >= 3), ['name','total_events_all_time','player_id','wppr_rank','ratings_value']]
player_count = mfp_df.loc[(mfp_df.weeks_played >= 5), ['name','total_events_all_time','player_id','wppr_rank','ratings_value']].shape[0]
player_count

46

In [31]:
mfp_wppr_projection = mfp_df.loc[(mfp_df.weeks_played >= 5), ['name','proj_score','proj_score_att','current_score','weeks_played','wppr_rank','ratings_value','current_wppr_value','score_to_beat','tva_total']]

In [32]:
# Final League Rankings
# Takes the minimum rank during the event of a tie, normally it would take the median rank
mfp_wppr_projection['league_rank'] = mfp_wppr_projection.current_score.rank(method='min',ascending=False)

In [33]:
# Linear Distribution based on number of players
# (PlayerCnt + 1 – Finishing Position) * 10/100 * (1st place value  / playerCnt)
mfp_wppr_projection['lin_wpprs'] = (player_count + 1 - mfp_wppr_projection['league_rank']) * 10/100 * (first_place_value/player_count)

In [34]:
# Dynamic Distribution
# (power(( 1 – power((( Finishing Position -1) / min(RatedPlayerCnt/2,64)),.7)),3)) * 90 /100 * (1st place value)
mfp_wppr_projection['dyn_wpprs'] = (pow((1 - pow(((mfp_wppr_projection.league_rank - 1) / min([player_count/2,64])),.7)),3) * 90/100 * first_place_value)
mfp_wppr_projection.loc[mfp_wppr_projection.dyn_wpprs < 0, 'dyn_wpprs'] = 0

In [35]:
mfp_wppr_projection['wppr_projection'] = mfp_wppr_projection.lin_wpprs + mfp_wppr_projection.dyn_wpprs

In [36]:
mfp_wppr_projection['new_wpprs'] = mfp_wppr_projection.current_wppr_value + mfp_wppr_projection.wppr_projection
mfp_wppr_projection['cur_wppr_league_rank'] = mfp_wppr_projection.current_wppr_value.rank(method='min',ascending=False)
mfp_wppr_projection['new_wppr_rank'] = mfp_wppr_projection.new_wpprs.rank(method='min',ascending=False)
mfp_wppr_projection['wppr_rank_change'] = mfp_wppr_projection.cur_wppr_league_rank - mfp_wppr_projection.new_wppr_rank

In [37]:
mfp_wppr_projection.sort_values('league_rank',inplace=True)
mfp_wppr_projection.loc[:,['league_rank','name','proj_score_att','current_score','score_to_beat','wppr_rank','wppr_projection', 'new_wpprs', 'wppr_rank_change','tva_total']]


,league_rank,name,proj_score_att,current_score,score_to_beat,wppr_rank,wppr_projection,new_wpprs,wppr_rank_change,tva_total
0,1.0,Cary Carmichael,175.200000,174.0,25.0,1006.0,26.078524,140.418524,1.0,0.090418
1,2.0,Xavier Marin,169.000000,169.0,24.0,355.0,19.020646,275.870646,0.0,0.446412
2,3.0,Michael Mattsson,167.888889,167.0,25.0,244.0,15.391445,323.441445,0.0,0.482155
3,4.0,John Tracey,164.000000,164.0,25.0,311.0,12.728154,286.388154,0.0,0.391537
4,5.0,Will Chernetsky,161.500000,161.0,24.0,1384.0,10.643042,94.393042,0.0,0.095373
5,6.0,Matthew Talley,160.000000,160.0,25.0,435.0,8.961905,234.791905,0.0,0.366337
6,7.0,Nate King,154.000000,154.0,21.0,1656.0,7.584937,76.154937,1.0,0.097380
7,7.0,Harry Franklin,154.000000,154.0,23.0,2476.0,7.584937,48.644937,3.0,0.062768
8,7.0,Eric Garcia,154.000000,154.0,21.0,1483.0,7.584937,85.314937,0.0,0.104910
9,10.0,Philip Priddy,153.800000,153.0,22.0,1073.0,4.717475,112.077475,0.0,0.121366


In [38]:
mfp_df[mfp_df.weeks_played == 4].loc[:,['name','wppr_rank','total_events_all_time']]

,name,wppr_rank,total_events_all_time
45,Dean Roblee,12405.0,2.0
47,Mitch Fadem,7936.0,4.0
48,Bobby Reagan,5342.0,5.0
49,Caitlin Campbell,7743.0,12.0
50,Jim Dismukes,27756.0,1.0


In [39]:
mfp_df[mfp_df.total_events_all_time == 4].loc[:,['name','wppr_rank','total_events_all_time']]

,name,wppr_rank,total_events_all_time
44,Glenn Howard-Erevia,10777.0,4.0
47,Mitch Fadem,7936.0,4.0
